# Notebook 05 - Tuning des hyperparam-tres

Objectif : r-gler les hyperparam-tres du meilleur couple mod-le-strat-gie identifi- dans `04_modeling.ipynb`, puis sauvegarder le mod-le final temporaire et les r-sultats de recherche.

## 0. Imports et configuration

In [ ]:
from pathlib import Path
import sys
import importlib.util
import pandas as pd
import joblib
from IPython.display import display

PROJECT_ROOT = Path("..").resolve() if Path.cwd().name == "notebooks" else Path(".").resolve()
MODULE_PATH = PROJECT_ROOT / "src" / "ml_phase3.py"
spec = importlib.util.spec_from_file_location("ml_phase3", MODULE_PATH)
ml_phase3 = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ml_phase3)

load_dataset = ml_phase3.load_dataset
make_splits = ml_phase3.make_splits
cross_validate_configurations = ml_phase3.cross_validate_configurations
modeling_configurations = ml_phase3.modeling_configurations
tune_estimator = ml_phase3.tune_estimator
model_score = ml_phase3.model_score
predict_with_threshold = ml_phase3.predict_with_threshold
metrics_from_scores = ml_phase3.metrics_from_scores
save_json = ml_phase3.save_json

pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")
MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)

## 1. R-cup-ration du meilleur mod-le issu du modeling

In [ ]:
df = load_dataset()
splits = make_splits(df)
X_train, y_train = splits["X_train"], splits["y_train"]
X_val, y_val = splits["X_val"], splits["y_val"]
X_train_val, y_train_val = splits["X_train_val"], splits["y_train_val"]

results_path = MODELS_DIR / "modeling_results.csv"
if results_path.exists():
    modeling_results = pd.read_csv(results_path)
else:
    modeling_results = cross_validate_configurations(X_train, y_train, modeling_configurations(), n_splits=5)
    modeling_results.to_csv(results_path, index=False)

best = modeling_results.iloc[0]
model_key = best["model"]
strategy = best["strategy"]
print(f"Mod?le ? tuner : {model_key} | strat?gie : {strategy}")
display(modeling_results.head(5))

## 2. Recherche d'hyperparam-tres

Les plages sont volontairement limit-es : assez larges pour tester le compromis biais-variance, mais raisonnables pour rester ex-cutables sur une machine -tudiante.

In [ ]:
best_estimator, tuning_results = tune_estimator(model_key, strategy, X_train, y_train)
tuning_results.to_csv(MODELS_DIR / "tuning_results.csv", index=False)
display(tuning_results[["rank_test_score", "mean_test_score", "std_test_score", "params"]].head(10))
print("Meilleur estimateur :")
print(best_estimator.named_steps["model"])

## 3. -valuation validation apr-s tuning

In [ ]:
scores_val = model_score(best_estimator, X_val)
y_pred_val = predict_with_threshold(best_estimator, X_val)
val_metrics = metrics_from_scores(y_val, scores_val, y_pred_val)
display(pd.DataFrame([val_metrics]))

## 4. Entra-nement final provisoire sur train + validation

Le seuil optimal sera choisi dans `06_evaluation.ipynb`, mais le pipeline entra-n- est d-j- sauvegard- pour assurer la reproductibilit-.

In [ ]:
# Le best_estimator retourn? par GridSearchCV/RandomizedSearchCV est d?j? refitt? sur X_train.
# On ne r?entra?ne pas ici sur la validation, car elle sert ? choisir le seuil dans 06_evaluation.
final_estimator = best_estimator
joblib.dump(final_estimator, MODELS_DIR / "tuned_model.joblib")

metadata = {
    "model_key": model_key,
    "strategy": strategy,
    "best_model": str(final_estimator.named_steps["model"]),
    "validation_metrics_default_threshold": val_metrics,
    "note": "Mod?le entra?n? sur train uniquement; validation r?serv?e au seuil de d?cision.",
}
save_json(metadata, MODELS_DIR / "tuning_metadata.json")
print("Mod?le tun? sauvegard? : models/tuned_model.joblib")

## 5. Synth-se tuning

Ce notebook produit :
- `models/tuning_results.csv`
- `models/tuned_model.joblib`
- `models/tuning_metadata.json`

La prochaine -tape est l'-valuation finale unique sur test set et le choix du seuil m-tier.